# Module 14 — APIs & Web Scraping (Phase 1: Topics 9–10)

Real data rarely arrives as a clean CSV. You fetch it from **APIs** (structured,
preferred) or **scrape** it from web pages (last resort). Your Kenya Economic Pulse
project pulled from the World Bank API — this module is the foundation under that.

Goals:
- What an API is; HTTP methods, status codes, JSON, params, headers, auth.
- Use `requests` correctly (with error handling & rate-limit awareness).
- Parse HTML with **BeautifulSoup** (fully reproducible on a local page here).
- Scrape **ethically & legally** (robots.txt, ToS, throttling).
- Turn messy responses into a tidy DataFrame.

## 14.1 What is an API?

An **API** (Application Programming Interface) lets your code request data from a
server. A **REST** API works over HTTP:

- **GET** — read data (most common for us). **POST** — send/create data.
- The server replies with a **status code** and (usually) a **JSON** body.
- **Status codes**: `200` OK · `201` created · `400` bad request · `401/403`
  auth problem · `404` not found · `429` too many requests (rate-limited) ·
  `500` server error.
- **Query params** refine the request (`?country=KE&year=2020`); **headers** carry
  metadata like an **API key** for auth.

Think of it as a restaurant: you (client) order from a menu (endpoints), the
kitchen (server) returns your dish (JSON).

In [1]:
import requests
import pandas as pd
import json

# We call a real, key-free public API (World Bank), but wrap it so the notebook
# STILL RUNS if the sandbox has no internet — falling back to a saved sample.
URL = "https://api.worldbank.org/v2/country/KE/indicator/NY.GDP.MKTP.CD"
params = {"format": "json", "date": "2015:2020", "per_page": 100}

def safe_get(url, params=None, timeout=8):
    try:
        r = requests.get(url, params=params, timeout=timeout)
        r.raise_for_status()          # raises on 4xx/5xx
        return r.json(), r.status_code, True
    except Exception as e:
        print(f"[offline fallback] network call failed ({type(e).__name__}); "
              f"using embedded sample data.")
        return None, None, False

payload, status, online = safe_get(URL, params)
print("online:", online, "| status:", status)

online: True | status: 200


In [2]:
# World Bank returns [metadata, data]. Provide a small embedded sample as fallback.
sample = [
    {"date": "2020", "value": 100380000000},
    {"date": "2019", "value": 100550000000},
    {"date": "2018", "value": 92210000000},
    {"date": "2017", "value": 82040000000},
    {"date": "2016", "value": 74820000000},
    {"date": "2015", "value": 70120000000},
]

if online and isinstance(payload, list) and len(payload) == 2 and payload[1]:
    records = payload[1]
    rows = [{"year": int(d["date"]), "gdp_usd": d["value"]} for d in records
            if d.get("value") is not None]
else:
    rows = [{"year": int(d["date"]), "gdp_usd": d["value"]} for d in sample]

gdp = pd.DataFrame(rows).sort_values("year").reset_index(drop=True)
gdp["gdp_usd_bn"] = (gdp["gdp_usd"] / 1e9).round(1)
print("Kenya GDP (current US$) — tidy DataFrame from a JSON API:")
gdp

Kenya GDP (current US$) — tidy DataFrame from a JSON API:


,year,gdp_usd,gdp_usd_bn
0,2015,7.012045e+10,70.1
1,2016,7.481514e+10,74.8
2,2017,8.203651e+10,82.0
3,2018,9.220298e+10,92.2
4,2019,1.003784e+11,100.4
5,2020,1.006575e+11,100.7


**What just happened:** we sent a GET with query params, checked the status, parsed
JSON, and flattened the nested structure into a tidy DataFrame — the exact loop
behind your Economic Pulse dashboard. Note the **error handling**: production code
never assumes the network works.

## 14.2 Reading API docs (the real skill)

You'll spend more time reading docs than coding. For any API, find:
1. **Base URL** and **endpoints** (what data is available).
2. **Auth**: none / API key in header / OAuth token.
3. **Parameters**: filtering, date ranges, fields.
4. **Pagination**: how to get page 2, 3, … (`per_page`, `offset`, `cursor`).
5. **Rate limits**: requests/min — respect them or get `429`.

Pattern for a key + pagination (pseudocode you can adapt):
```python
headers = {"Authorization": f"Bearer {API_KEY}"}
all_rows, page = [], 1
while True:
    r = requests.get(url, params={"page": page, "per_page": 100}, headers=headers)
    data = r.json()
    if not data: break
    all_rows.extend(data); page += 1
    time.sleep(0.5)            # be polite: throttle between requests
```

## 14.3 Web scraping with BeautifulSoup

When there's **no API**, you parse HTML. We use a local HTML string here so the
lesson is 100% reproducible, but the parsing code is identical for a real
`requests.get(url).text`.

In [3]:
from bs4 import BeautifulSoup

# Imagine this came from: html = requests.get(url).text
html = '''
<html><body>
  <h1>Top Stocks</h1>
  <table id="stocks">
    <tr><th>Ticker</th><th>Price</th><th>Sector</th></tr>
    <tr><td class="tk">SCOM</td><td class="pr">18.50</td><td>Telecom</td></tr>
    <tr><td class="tk">EQTY</td><td class="pr">45.20</td><td>Banking</td></tr>
    <tr><td class="tk">KCB</td><td class="pr">38.75</td><td>Banking</td></tr>
    <tr><td class="tk">EABL</td><td class="pr">155.00</td><td>Consumer</td></tr>
  </table>
  <a href="/page/2">Next</a>
</body></html>
'''

soup = BeautifulSoup(html, "html.parser")
print("page title (h1):", soup.find("h1").text)
print("the 'Next' link:", soup.find("a")["href"])

page title (h1): Top Stocks
the 'Next' link: /page/2


In [4]:
# Extract the table into a DataFrame — the core scraping task
rows = []
for tr in soup.select("table#stocks tr")[1:]:      # skip header row
    cells = tr.find_all("td")
    rows.append({
        "ticker": cells[0].text.strip(),
        "price":  float(cells[1].text.strip()),
        "sector": cells[2].text.strip(),
    })

stocks = pd.DataFrame(rows)
print(stocks)
print("\naverage price by sector:")
print(stocks.groupby("sector")["price"].mean().round(2))

  ticker   price    sector
0   SCOM   18.50   Telecom
1   EQTY   45.20   Banking
2    KCB   38.75   Banking
3   EABL  155.00  Consumer

average price by sector:
sector
Banking      41.98
Consumer    155.00
Telecom      18.50
Name: price, dtype: float64


**Selectors you'll use most:**
- `soup.find("tag")` — first match; `soup.find_all("tag")` — all matches.
- `soup.select("css selector")` — powerful CSS querying (`#id`, `.class`, `a > b`).
- `.text` / `.get_text()` — the visible text; `element["attr"]` — an attribute.

**Tip:** many tables can be grabbed in one line with `pd.read_html(html)` — try it
before hand-parsing.

In [5]:
# pandas can often read HTML tables directly:
tables = pd.read_html(html)          # returns a list of DataFrames
print("pd.read_html found", len(tables), "table(s):")
tables[0]

pd.read_html found 1 table(s):


/tmp/ipykernel_4786/2163686717.py:2: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(html)          # returns a list of DataFrames


,Ticker,Price,Sector
0,SCOM,18.50,Telecom
1,EQTY,45.20,Banking
2,KCB,38.75,Banking
3,EABL,155.00,Consumer


## 14.4 Scrape ethically and legally (say this in interviews)

- **Check `robots.txt`** (`site.com/robots.txt`) and the site's **Terms of
  Service** — some forbid scraping.
- **Throttle**: add `time.sleep()`; never hammer a server (that's abusive and can
  get you IP-banned).
- **Identify yourself** with a `User-Agent` header; consider caching responses.
- **Prefer an API** if one exists — it's more stable and permitted.
- Don't scrape **personal/copyrighted** data you have no right to use.

> "I always check robots.txt and ToS, prefer an official API, throttle my requests,
> and cache results so I never re-hit a server unnecessarily."

## 14.5 Mini-exercises

1. Modify `safe_get` to retry up to 3 times with a short `time.sleep` on failure.
2. From the stocks table, scrape only **Banking** rows into a DataFrame.
3. Given the `<a href="/page/2">`, write the loop logic to follow pagination
   (pseudocode is fine).
4. Explain the difference between using an API and scraping, and when you'd choose
   each.

## Summary

- APIs return structured **JSON**; use `requests.get(url, params, headers)`, check
  **status codes**, and **handle errors**. Mind **auth**, **pagination**,
  **rate limits**.
- **Scrape** only when there's no API; parse with **BeautifulSoup** (`find`,
  `select`) or `pd.read_html`; then tidy into a DataFrame.
- Always scrape **ethically/legally**: robots.txt, ToS, throttle, identify, cache.

Next: **Module 15 — Probability, Combinatorics & Bayes** (the math of uncertainty).